# HydroSafe AI - Member 2: Data Validation Agent

**Owner:** Member 2 (validation)  
**Dataset:** `Hydrometeorological_data_2019-2026.csv` (2,800 records, daily)  
**Status:** Complete for the validation agent

## Scope (strictly Member 2)

This notebook delivers exactly one thing: a deterministic **data quality and sensor health validation** step. It returns the frozen fields the lead orchestrator and the risk agent (Member 6) consume:

| Contract field    | Where it lives in the output          |
|-------------------|---------------------------------------|
| `score`           | `report["score"]`                     |
| `valid count`     | `report["valid_count"]`               |
| `flagged count`   | `report["flagged_count"]`             |
| `sensor health`   | `report["sensor_health"]`             |

## Explicitly OUT of scope
- No anomaly detection (Member 4), no trends (Member 3), no correlations (Member 5).
- No risk scoring (Member 6) and no orchestration/routes/database (Member 1).
- The agent never infers physical dam risk. Poor data quality only lowers the score and raises a validation warning, exactly as the integration contract requires.

## Input description

Columns in the raw file (with observed behaviour in the 2019-2026 file):

| Column                  | Observed range     | Known data issues                          |
|-------------------------|--------------------|--------------------------------------------|
| `Date`                  | MM/DD/YYYY HH:MM:SS| First field is the month (never > 12)      |
| `Reservoir water level` | 390.78 - 461.34    | Missing for the first 120 rows             |
| `Tail water level`      | 385.36 - 399.29    | Missing for the first 120 rows             |
| `In flow`               | 86.5 - 2730        | 2 readings sit within +-0.5 m of tail level|
| `Daily mean temperature`| 4.5 - 35.4         | One missing value in the final row         |
| `Daily rainfall`        | 0 - 139            | None                                       |

The pipeline runs the following deterministic checks:
1. Timestamp is parseable and not duplicated.
2. Every sensor reading is present and numeric.
3. Every reading is inside its physical bound (no negatives; temperature between -50 and 60).
4. Hydraulic consistency: `Tail water level <= Reservoir water level + 0.5 m` tolerance.
5. Completeness per sensor (missing ratio) drives the `sensor_health` status:
   `HEALTHY <= 2%`, `WATCH <= 10%`, otherwise `POOR`.

In [ ]:
import json
import math
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("HydroSafe AI - Member 2 imports loaded.")

In [ ]:
import glob
from google.colab import files

candidates = [p for p in glob.glob("/content/*.csv")]
if candidates:
    csv_path = candidates[0]
    print("Found in /content:", os.path.basename(csv_path))
else:
    print("No CSV found in /content yet - upload Hydrometeorological_data_2019-2026.csv now.")
    uploaded = files.upload()
    csv_path = list(uploaded)[0]

df = pd.read_csv(csv_path)
display(df.head())
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
print(df.describe().T)

## Validation configuration and public function

Per the repository rules, every agent exposes **one public function with a stable signature**. This notebook therefore defines exactly one: `validate_data(data, config=None) -> dict`.

All thresholds live in `VALIDATION_CONFIG` so the lead or Member 2 can tune them without touching the frontend or the contract.

In [ ]:
DATE_COLUMN = "Date"

SENSOR_COLUMNS = {
    "reservoir_water_level": {"source": "Reservoir water level", "min": 0.0, "max": None},
    "tail_water_level": {"source": "Tail water level", "min": 0.0, "max": None},
    "in_flow": {"source": "In flow", "min": 0.0, "max": None},
    "daily_mean_temperature": {"source": "Daily mean temperature", "min": -50.0, "max": 60.0},
    "daily_rainfall": {"source": "Daily rainfall", "min": 0.0, "max": None},
}

VALIDATION_WEIGHTS = {"completeness": 0.5, "validity": 0.5}
HEALTHY_MAX_MISSING_RATIO = 0.02
WATCH_MAX_MISSING_RATIO = 0.10
RELATION_TOLERANCE_M = 0.5


def _empty_report(df, message):
    return {
        "agent": "member2_validation",
        "status": "ok_with_warning",
        "score": 0.0,
        "total_records": 0 if df is None else len(df),
        "valid_count": 0,
        "flagged_count": 0 if df is None else len(df),
        "quality_status": "POOR",
        "sensor_health": {},
        "flags": [],
        "warning": message,
    }

In [ ]:
def validate_data(data, config=None):
    """Public Member 2 entry point (stable signature).

    Runs deterministic data-quality checks over a reservoir sensor frame
    and returns the frozen validation contract consumed by the lead
    orchestrator. Never infers physical risk.
    """
    if data is None or len(data) == 0:
        return _empty_report(data, "Empty input: no records received for validation.")

    df = data.copy()
    missing_sources = [spec["source"] for spec in SENSOR_COLUMNS.values()
                       if spec["source"] not in df.columns]
    if missing_sources:
        return _empty_report(df, "Expected sensor columns are missing from input: %s." % missing_sources)
    if DATE_COLUMN not in df.columns:
        return _empty_report(df, "Expected timestamp column '%s' is missing." % DATE_COLUMN)

    total = len(df)
    timestamps = pd.to_datetime(df[DATE_COLUMN], errors="coerce")
    timestamp_ok = timestamps.notna()
    duplicated = timestamps[timestamps.duplicated(keep=False)].index.tolist()

    flags = []
    for idx in df.index:
        stamp = timestamps.at[idx]
        stamp_text = (stamp.strftime("%Y-%m-%d %H:%M:%S")
                      if pd.notna(stamp) else str(df.at[idx, DATE_COLUMN]))

        if not bool(timestamp_ok.at[idx]):
            flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": DATE_COLUMN,
                          "reason": "invalid_datetime",
                          "detail": "Value could not be parsed as a timestamp."})

        for name, spec in SENSOR_COLUMNS.items():
            raw = df.at[idx, spec["source"]]
            num = pd.to_numeric(pd.Series([raw]), errors="coerce").iloc[0]
            if pd.isna(num) and pd.isna(raw):
                flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": name,
                              "reason": "missing",
                              "detail": "Empty %s reading." % name.replace("_", " ")})
            elif pd.isna(num):
                flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": name,
                              "reason": "not_numeric",
                              "detail": "Non-numeric value %r." % raw})
            else:
                if num < spec.get("min", -math.inf):
                    flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": name,
                                  "reason": "below_minimum",
                                  "detail": "%s is below minimum %s." % (str(num), spec["min"])})
                if spec.get("max") is not None and num > spec["max"]:
                    flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": name,
                                  "reason": "above_maximum",
                                  "detail": "%s is above maximum %s." % (str(num), spec["max"])})

        res = pd.to_numeric(pd.Series([df.at[idx, "Reservoir water level"]]), errors="coerce").iloc[0]
        tail = pd.to_numeric(pd.Series([df.at[idx, "Tail water level"]]), errors="coerce").iloc[0]
        if pd.notna(res) and pd.notna(tail) and tail > res + RELATION_TOLERANCE_M:
            flags.append({"row": int(idx), "timestamp": stamp_text, "sensor": "tail_water_level",
                          "reason": "implausible_relation",
                          "detail": "Tail water level %s exceeds reservoir level %s beyond the %s m tolerance."
                          % (str(tail), str(res), RELATION_TOLERANCE_M)})

    for dup in duplicated:
        flags.append({"row": int(dup), "timestamp": str(df.at[dup, DATE_COLUMN]),
                      "sensor": DATE_COLUMN, "reason": "duplicate_timestamp",
                      "detail": "Timestamp appears more than once."})

    sensor_health = {}
    for name, spec in SENSOR_COLUMNS.items():
        col = df[spec["source"]]
        numeric = pd.to_numeric(col, errors="coerce")
        missing = int(col.isna().sum())
        non_numeric = int((numeric.isna() & col.notna()).sum())
        below = int(((numeric < spec.get("min", -math.inf)) & numeric.notna()).sum())
        upper = spec.get("max")
        above = int((numeric > upper).sum()) if upper is not None else 0
        out_of_range = below + above
        missing_ratio = round(missing / total, 4) if total else 0.0
        if missing_ratio <= HEALTHY_MAX_MISSING_RATIO:
            status = "HEALTHY"
        elif missing_ratio <= WATCH_MAX_MISSING_RATIO:
            status = "WATCH"
        else:
            status = "POOR"
        if (non_numeric + out_of_range) > 0 and status == "HEALTHY":
            status = "WATCH"
        sensor_health[name] = {
            "total": total,
            "missing": missing,
            "missing_ratio": missing_ratio,
            "non_numeric": non_numeric,
            "out_of_range": out_of_range,
            "status": status,
        }

    flagged_rows = sorted({f["row"] for f in flags})
    flagged_count = len(flagged_rows)
    valid_count = total - flagged_count

    completeness = ((sum(1.0 - s["missing_ratio"] for s in sensor_health.values())
                     / len(sensor_health)) if sensor_health else 0.0)
    validity = valid_count / total if total else 0.0
    score = round(100.0 * (VALIDATION_WEIGHTS["completeness"] * completeness
                           + VALIDATION_WEIGHTS["validity"] * validity), 1)

    if score >= 90:
        quality_status = "GOOD"
    elif score >= 70:
        quality_status = "WATCH"
    else:
        quality_status = "POOR"

    warning = None
    if total and flagged_count:
        worst_sensor = max(sensor_health, key=lambda key: sensor_health[key]["missing_ratio"])
        warning = ("%s of %s records (%s%%) are flagged; sensor '%s' carries the most missing readings."
                   % (flagged_count, total, round(100.0 * flagged_count / total, 1), worst_sensor))
    elif score < 70:
        warning = ("Quality score below 70. Downstream confidence must be reduced and the pipeline "
                   "must avoid escalating physical risk from unreliable data.")

    return {
        "agent": "member2_validation",
        "status": "ok" if score >= 90 else "ok_with_warning",
        "score": score,
        "total_records": total,
        "valid_count": valid_count,
        "flagged_count": flagged_count,
        "quality_status": quality_status,
        "sensor_health": sensor_health,
        "flags": flags,
        "warning": warning,
    }

In [ ]:
report = validate_data(df)

print("Agent           :", report["agent"])
print("Status          :", report["status"])
print("Quality score   :", report["score"])
print("Quality status  :", report["quality_status"])
print("Total records   :", report["total_records"])
print("Valid records   :", report["valid_count"])
print("Flagged records :", report["flagged_count"])
print("Warning         :", report["warning"])
print()
display(pd.DataFrame(report["sensor_health"]).T)
print()
if report["flags"]:
    display(pd.DataFrame(report["flags"]).head(15))
    print("(showing the first 15 of %d flags)" % len(report["flags"]))

In [ ]:
if report["sensor_health"]:
    names = list(report["sensor_health"])
    ratios = [report["sensor_health"][n]["missing_ratio"] * 100 for n in names]
    colors = ["#2ca02c" if report["sensor_health"][n]["status"] == "HEALTHY"
              else ("#ff7f0e" if report["sensor_health"][n]["status"] == "WATCH" else "#d62728")
              for n in names]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(names, ratios, color=colors)
    ax.axhline(2, color="grey", linestyle="--", linewidth=1, label="HEALTHY <= 2%")
    ax.axhline(10, color="grey", linestyle=":", linewidth=1, label="WATCH <= 10%")
    ax.set_ylabel("Missing ratio (%)")
    ax.set_title("Sensor completeness - Member 2 validation")
    ax.legend()
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

## Reading the results (2019-2026 file)

Expected behaviour on the supplied dataset:

- `Reservoir water level` and `Tail water level` show roughly 4.3% missing readings (the first 120 rows), so both report **WATCH** sensor health.
- Two readings on 11/07/2019 and 11/08/2019 have tail level nominally above reservoir level, but they stay inside the 0.5 m tolerance, so they remain unflagged by design.
- The final 09/01/2026 row has no temperature reading, so one `Daily mean temperature` value is missing.
- Overall quality stays **GOOD** (score around 97) because the dataset is otherwise clean.

The `warning` text informs Members 1 and 6 that flagged records exist; the risk agent should treat that as a lower-confidence signal, never as a reason to fabricate physical risk.

In [ ]:
def _good_frame(n=60, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    return pd.DataFrame({
        DATE_COLUMN: pd.date_range("2024-01-01", periods=n, freq="D"),
        "Reservoir water level": np.round(430 + np.sin(t / 7) * 3, 2),
        "Tail water level": np.round(392 + np.sin(t / 9) * 1, 2),
        "In flow": np.round(450 + rng.normal(0, 30, n), 2).clip(min=0),
        "Daily mean temperature": np.round(20 + np.cos(t / 11) * 5, 2),
        "Daily rainfall": np.round(rng.random(n) * 20, 2).clip(min=0),
    })


good = _good_frame()
r1 = validate_data(good)
assert r1["flagged_count"] == 0, r1
assert r1["valid_count"] == len(good), r1
assert r1["score"] >= 99, r1
print("1. Clean synthetic frame -> flagged=%s valid=%s score=%s"
      % (r1["flagged_count"], r1["valid_count"], r1["score"]))

bad = good.copy()
bad["In flow"] = bad["In flow"].astype(object)
bad.loc[0, "Reservoir water level"] = np.nan
bad.loc[1, "In flow"] = "abc"
bad.loc[2, "Daily rainfall"] = -5
bad.loc[3, "Tail water level"] = bad.loc[3, "Reservoir water level"] + 1000.0
bad.loc[4, DATE_COLUMN] = bad.loc[3, DATE_COLUMN]
r2 = validate_data(bad)
assert r2["flagged_count"] >= 4, r2
reasons = {f["reason"] for f in r2["flags"]}
assert "implausible_relation" in reasons, reasons
assert r2["score"] < r1["score"], (r1, r2)
print("2. Corrupted frame -> flagged=%s score=%s reasons=%s"
      % (r2["flagged_count"], r2["score"], sorted(reasons)))

r3 = validate_data(pd.DataFrame())
assert r3["quality_status"] == "POOR" and r3["score"] == 0.0, r3
print("3. Empty frame -> status=%s score=%s (no crash)" % (r3["quality_status"], r3["score"]))

r4 = validate_data(good.copy().iloc[:, :0])
assert r4["quality_status"] == "POOR", r4
print("4. Frame with no columns -> status=%s (no crash)" % r4["quality_status"])

print("Poor data quality lowers the score: %.1f -> %.1f" % (r1["score"], r2["score"]))
assert r2["score"] < r1["score"]
print("All Member 2 unit checks passed.")

In [ ]:
with open("member2_validation_report.json", "w") as fh:
    json.dump(report, fh, indent=2)
print("Saved member2_validation_report.json (%d bytes)" % os.path.getsize("member2_validation_report.json"))

try:
    from google.colab import files
    files.download("member2_validation_report.json")
except Exception:
    print("Run inside Google Colab to auto-download the report file.")

In [ ]:
print(json.dumps(report, indent=2, ensure_ascii=False))